In [1]:
#Import needed libraries
#holtwinters in stats model is used for smoothing

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt
from sklearn.linear_model import LinearRegression

%matplotlib inline

In [2]:
#Load the data with tab-separated values (not comma separated data set), so we add sep='\t'
file_path = 'electricity demand.csv'
data = pd.read_csv(file_path, sep='\t')

#Parse the data into a time series format from string format (this block is conversion code to change word month to date)
data['Date'] = pd.date_range(start='1/1/2001', periods=len(data), freq='ME')
data.set_index('Date', inplace=True) #Using the date as the index
data.index.freq = 'ME' #Explicitly set the frequency because we are using exponential smoothing. This suppresses the python error
data_series = data ['KWh']

In [3]:
#Define a function to calculate error metrics
def calculate_metrics(actual, forecast):
    mae = mean_absolute_error(actual, forecast) 
    mse = mean_squared_error(actual, forecast)
    rmse = np.sqrt(mse) #Root Mean Squared Error
    mape = np.mean(np.abs((actual - forecast) / actual)) * 100
    return mape, mae, mse, rmse

In [4]:
#Extend the index to include the next 6 months. We have 3 mo's of data. Asking to predict 6 add'l months in the future
#Creating an array {future_dates) that has 6 additional lines in it (6 periods)
future_dates = pd.date_range(start=data_series.index[-1] + pd.offsets.MonthBegin(), periods=6, freq='ME')
#This is all the PRE-PROCESSING WE HAVE TO DO

In [6]:
#3-Month Moving Average
#We have to tell it how many pieces of data we will process for this prediction (window = 3) | Average them together (mean)
moving_avg_3 = data_series.rolling(window=3).mean()

#moving_avg_3_future = np.mean(data_series[-3:]) #Using the mean of the last 3 months for future predictions | create EMPTY SERIES
#We have to build a space to put the three new predictions
moving_avg_3_future = []

#creating a copy of the data set b/c we will reference it again | -3 MEANS THREE DATA POINTS BEFORE IT
extended_series = data_series.copy()
for _ in range(6):
    next_value = extended_series[-3:].mean() #Creating a loop and defining what the next value is (next value is mean of 3 pieces of data before it)
    moving_avg_3_future.append(next_value) #Adding next value to the end of the future data in the empty series we built for new predictions
    extended_series = pd.concat([extended_series, pd.Series([next_value], index=[extended_series.index[-1] + pd.offsets.MonthBegin()])])


In [ ]:
#Added code so we can see what the predictions are in the upcoming months prior to completing the (small loop)
#printing month {iteration} prediction: float variable with 2 decimal points

In [7]:
for i, prediction in enumerate(moving_avg_3_future, 1):
    print(f"Month {i} prediction: {prediction:.2f}") #i = iteration

Month 1 prediction: 82.67
Month 2 prediction: 81.89
Month 3 prediction: 81.52
Month 4 prediction: 82.02
Month 5 prediction: 81.81
Month 6 prediction: 81.78


In [8]:
#UPDATED: Iterative calculation for future values using pd.concat
weighted_moving_avg_3_future = []
extended_series = data_series.copy()

#3-Month WEIGHTED Moving Average without Lambda function
weights = np.array([0.5, 0.3, 0.2])
def weighted_moving_average(series, window, weights): #series - where data is, window - how long
    result=[] #creating empty array that will get filled later
    for i in range(len(series)): #this is the LOOP
        if i + 1 < window:
            result.append(np.nan) #when it gets to the end, it will add an additional line & fill it with not a number (nan)
        else:
            weighted_sum = 0
            for j in range(window):
                weighted_sum += series.iloc[i - j] * weights[j]
            result.append(weighted_sum)
    return pd.Series(result, index=series.index)

weighted_moving_avg_3 = weighted_moving_average(data_series, 3, weights)



In [9]:
for i, prediction in enumerate(weighted_moving_avg_3_future, 1):
    print(f"Month {i} weighted prediction: {prediction:.2f}")

In [10]:
#Exponential Smoothing with a smoothing constant of 0.2 using HOLT
exp_smoothing_0_2_model = Holt(data_series, initialization_method="estimated").fit(smoothing_level=0.2)
exp_smoothing_0_2 = exp_smoothing_0_2_model.fittedvalues
exp_smoothing_0_2_future = exp_smoothing_0_2_model.forecast(6)

In [11]:
for i, prediction in enumerate(exp_smoothing_0_2_future, 1):
    print(f"Month {i} exponential smoothing prediction: {prediction:.2f}")

Month 1 exponential smoothing prediction: 95.27
Month 2 exponential smoothing prediction: 95.54
Month 3 exponential smoothing prediction: 95.82
Month 4 exponential smoothing prediction: 96.09
Month 5 exponential smoothing prediction: 96.36
Month 6 exponential smoothing prediction: 96.64


In [12]:
#Exponential Smoothing with Smoothing constant of 0.6
exp_smoothing_0_6_model = Holt(data_series, initialization_method="estimated").fit(smoothing_level=0.6) #just change smoothing here
exp_smoothing_0_6 = exp_smoothing_0_6_model.fittedvalues
exp_smoothing_0_6_future = exp_smoothing_0_6_model.forecast(6)

In [13]:
for i, prediction in enumerate(exp_smoothing_0_6_future, 1):
    print (f"Month {i} exponential smoothing prediction: {prediction: .2f}")

Month 1 exponential smoothing prediction:  82.79
Month 2 exponential smoothing prediction:  82.91
Month 3 exponential smoothing prediction:  83.02
Month 4 exponential smoothing prediction:  83.14
Month 5 exponential smoothing prediction:  83.25
Month 6 exponential smoothing prediction:  83.37


In [14]:
#Simple Linear Regression | ONE Independent Var = KWh and Dependent Var = Month
months = np.arange(len(data_series)).reshape(-1, 1) # create a data series for the months
reg_model = LinearRegression().fit(months, data_series)
future_months = np.arange(len(data_series), len(data_series) + 6).reshape(-1, 1) #created an additional 6 spots where we can place predicted data
linear_regression = reg_model.predict(months) # predicted values
linear_regression_future = reg_model.predict(future_months) #add predicted values to linear regression future 

In [15]:
#get predictions for Jan, Feb, Mar, Apr, May, June
for i, prediction in enumerate(linear_regression_future, 1):
    print (f"Month {i} linear regression prediction: {prediction: .2f}")

Month 1 linear regression prediction:  100.84
Month 2 linear regression prediction:  101.21
Month 3 linear regression prediction:  101.58
Month 4 linear regression prediction:  101.95
Month 5 linear regression prediction:  102.32
Month 6 linear regression prediction:  102.69


In [16]:
#Calculate error metrics for each method
metrics = {
    '3-Month Moving Average': calculate_metrics(data_series[2:], moving_avg_3.dropna()), 
    '3-Month Weighted Moving Average': calculate_metrics(data_series[2:], weighted_moving_avg_3.dropna()), 
    'Exponential Smoothing 0.2': calculate_metrics(data_series[1:], exp_smoothing_0_2[1:]),
    'Exponential Smoothing 0.6': calculate_metrics(data_series[1:], exp_smoothing_0_6[1:]),
    'Linear Regression': calculate_metrics(data_series, linear_regression)
}

In [17]:
#Display metrics | Mean Avg Percentage Error, Mean Absolute Error, Mean Squared Error, and Root Mean Squared Error
metrics_df = pd.DataFrame(metrics, index=['MAPE', 'MAE', 'MSE', 'RMSE'])
metrics_df


,3-Month Moving Average,3-Month Weighted Moving Average,Exponential Smoothing 0.2,Exponential Smoothing 0.6,Linear Regression
MAPE,8.113069,5.770385,13.764714,11.654046,13.678865
MAE,7.990196,5.682353,13.095035,11.228212,12.932261
MSE,97.767974,50.644118,253.458544,194.433346,228.624346
RMSE,9.887769,7.116468,15.920381,13.943936,15.120329


In [24]:
#On all of these we are looking for the smallest values. We want to use the 3-Month Weighted Moving Average
